# cuda-torso + GBDT booster — Kaggle GPU

Runs the **leaderboard-winning engine** (cuda-torso: GPU per-threshold
neuroevolution) with an **additive GBDT front-booster** layered on, plus a
`--no_gbdt` **control** for a clean ablation. Everything is bundled in
`cuda_torso_gbdt.zip` (their `run.py` + `libeval.cu` + `run_gbdt.py` + all three
graphs).

**Setup:** Settings → Accelerator → **GPU T4 x2** (or P100), Internet **ON**.
**Add Data → Upload → `cuda_torso_gbdt.zip`.** Then Run All.

In [ ]:
# 1) locate the bundle (uploaded as a Kaggle dataset) and stage to writable dir
import os, glob, zipfile, shutil
shutil.rmtree('/kaggle/working/cuda', ignore_errors=True); os.makedirs('/kaggle/working/cuda', exist_ok=True)
z  = glob.glob('/kaggle/input/**/cuda_torso_gbdt.zip', recursive=True)
ex = glob.glob('/kaggle/input/**/run_gbdt.py', recursive=True)
if z:    zipfile.ZipFile(z[0]).extractall('/kaggle/working/cuda')
elif ex: shutil.copytree(ex[0].rsplit('/run_gbdt.py',1)[0], '/kaggle/working/cuda/cuda-torso-main')
else:    raise SystemExit("Upload cuda_torso_gbdt.zip via Add Data.")
ROOT = os.path.dirname(glob.glob('/kaggle/working/cuda/**/run_gbdt.py', recursive=True)[0])
os.chdir(ROOT)
print("project:", ROOT, "| files:", sorted(os.listdir('.')))

In [ ]:
# 2) compile the cuda-torso evaluator -> libeval.so  (one time)
!cd "/kaggle/working/cuda/cuda-torso-main" && nvcc -shared -Xcompiler -fPIC -o libeval.so libeval.cu && echo OK && ls -la libeval.so

In [ ]:
# 3) deps + GPU check (sklearn is used by the GBDT booster; usually preinstalled)
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
try:
    import sklearn; print("sklearn", sklearn.__version__)
except ImportError:
    import subprocess; subprocess.run(["pip","install","-q","scikit-learn"]); import sklearn; print("installed sklearn", sklearn.__version__)

## Smoke test on small-graph (confirm both modes behave)
Small is near-optimal, so the booster's gain will be tiny here — the point is to
confirm the engine reaches ~the optimum and the ablation runs cleanly. Stop each
cell after a few hundred generations (▢) once you've seen the `official` line move.

In [ ]:
# boosted: cuda-torso engine + GBDT front-booster
!cd /kaggle/working/cuda/cuda-torso-main && python3 run_gbdt.py --graph small-graph --gbdt_every 200 --seed 0

In [ ]:
# control: vanilla cuda-torso (identical engine, seed, budget; GBDT OFF)
!cd /kaggle/working/cuda/cuda-torso-main && python3 run_gbdt.py --graph small-graph --no_gbdt --seed 0

## The real run — medium or large (where there is headroom)
This is where a positive GBDT delta, and any push past the leader, can actually
happen. Let it run the full session; it checkpoints to
`submissions/<graph>/<score>.json` every 50 gens (filename = HVI; more negative =
better). Run the control too for the ablation.

In [ ]:
# boosted, long session
!cd /kaggle/working/cuda/cuda-torso-main && python3 run_gbdt.py --graph medium-graph --gbdt_every 200 --seed 0

In [ ]:
# control, long session (run in a second session or after, same seed)
!cd /kaggle/working/cuda/cuda-torso-main && python3 run_gbdt.py --graph medium-graph --no_gbdt --seed 0

## Collect the best front
The most-negative filename in `submissions/<graph>/` is the best. Download it and
re-score locally with `tools/portfolio.py` before quoting any number.

In [ ]:
import glob, os, shutil
G="medium-graph"  # change to the graph you ran
subs = glob.glob(f'submissions/{G}/*.json')
best = min(subs, key=lambda f:int(os.path.basename(f).split('.')[0])) if subs else None
print("checkpoints:", len(subs), "| best:", best)
if best:
    shutil.copy(best, f'/kaggle/working/{G}_best.json')
    print(f"saved /kaggle/working/{G}_best.json")
print("leaderboard targets: small -1,829,919 | medium -1,745,122 | large -5,493,062")